# Section 1 — contract destination starter

**Varshini: change destination only.** Roll timing, capacity handling, rounding, target weights, the common benchmark, accounting and metric stay fixed. Choose a destination only at a scheduled baseline roll event; do not reselect daily.

[Part 2 Issue #12](https://github.com/JacopoMichelacci/Futures_Class/issues/12), Version 8 §§39–42: next deferred (baseline), second deferred, closest to 45 sessions, closest to 75 sessions. These alternatives are TODOs, not implemented below.

Run all cells from a fresh Python kernel using the project's existing uv environment. These notebooks use only the shared synthetic fixture and standard-library Python, with no online access. Outputs are deliberately cleared before sharing.

**Part 2 responsibility:** each teammate implements their own reusable engine and substantive tests, opens their own issue-linked PR, and reviews teammate PRs. This starter supplies inputs and interface examples, not the strand solution. Keep engine logic in `src/` and tests in `tests/unit/`. No empirical conclusions or preferred-method selection belong here. Do not consume another strand's preferred results.

See [Section 0 interfaces](../docs/section0_phase_a.md) and [fixture provenance](../data/fixtures/README.md). Phase B still needs sourced first-notice metadata, verified exchange/shortened-session times, historical benchmark/baseline quantities, production tables, the common 100-date/70% gate and AUM/capacity preflight. The synthetic calendar is not a real CBOT calendar. Daily return Tracking RMSE in basis points remains the sole ranking metric for the later historical comparison.

### Environment and kernel: use the existing lockfile

Before opening a notebook, run these commands in the current checkout's repository root:

```bash
uv sync --locked
uv run --no-sync python -c "import sys, ipykernel, nbformat; print(sys.executable)"
```

The first command synchronises the existing shared lockfile; it may need network access to obtain packages. The second verifies the declared notebook packages and prints the interpreter to select. Notebook examples themselves run offline after setup. Do not independently add packages, use `uv add` or `pip install`, or edit `pyproject.toml`/`uv.lock`. If synchronisation or imports fail, report the error to Jacopo rather than changing dependencies.

In VS Code, use **Select Kernel → Python Environments** and select the interpreter printed above for this checkout (normally `.venv/bin/python`). With an existing Jupyter frontend, select a kernel backed by that same interpreter. If it is not listed, coordinate kernel setup with Jacopo; do not substitute an unrelated environment. Restart the kernel, start it in this repository or the notebook's folder, then run all cells. A separate worktree must use its intended project environment; do not reuse a kernel already importing another checkout.

In [ ]:
from pathlib import Path
from decimal import Decimal
from pprint import pprint
import sys

# Start the kernel in this repository or the notebook's own folder.
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "config/project.yaml").is_file()
             and (p / "data/fixtures/phase_a.json").is_file()
             and (p / "src/__init__.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the kernel in the repository or this notebook's folder.")
sys.path.insert(0, str(ROOT))
import src
assert Path(src.__file__).resolve().parent == ROOT / "src", "Restart the kernel for this checkout."
from src.validation.schemas import load_project_config, HOLDINGS_COLUMNS
from src.data.fixtures import load_fixture

config = load_project_config(ROOT / "config/project.yaml")
f = load_fixture(ROOT / "data/fixtures/phase_a.json",
                 config["max_liquidity_source_age_sessions"])
return_date = f.raw["return_dates"][0]
context = f.calendar.decision(return_date)
print("SYNTHETIC FIXTURES ONLY", config["roots"], "return interval ends", return_date)


### Inspect the shared baseline event and its decision-time curve

Use the shared generator to obtain the outgoing contract and roll date. The relevant curve below is from that event's decision session. Eligibility uses the supplied delivery-avoidance anchor; expiration is not substituted.

**Complete-curve precondition:** `build_baseline_path` assumes a verified complete eligible curve for each decision date, using only information available by that cut-off. Absence of the active contract from a supplied complete curve may represent confirmed loss of eligibility. An unexplained missing contract row is not equivalent to confirmed ineligibility and must not be passed through as an eligibility decision. Production data preparation must validate curve completeness before calling the state machine; the helper cannot distinguish a missing-data omission from a confirmed removal. The explicit synthetic baseline case is supplied as a complete hand fixture. No state-machine behaviour is changed here.

In [ ]:
from src.portfolio.baseline import build_baseline_path
case = f.raw["baseline_path_case"]
path = build_baseline_path(case["eligible_curve"], case["decision_dates"],
                           case["commodities"], f.calendar, f.catalogue)
event = path.roll_events[0]
pprint(event)
pprint(path.positions[:3])


In [ ]:
outgoing = f.catalogue[event["outgoing_contract"]]
event_curve = [r for r in case["eligible_curve"]
               if r["date"] == event["date"] and r["commodity"] == event["commodity"]]
# Inspection only: no alternative destination is selected here.
deferred_rows = [r for r in event_curve if r["eligible"] and r["anchor"] > outgoing.anchor]
pprint(sorted(deferred_rows, key=lambda r: (r["anchor"], r["contract"])))


### Expected rule output and work to complete

An illustrative destination record contains the unchanged decision/event date, commodity, outgoing contract, selected exact contract and an explicit fallback flag. This is a suggested adapter shape, not a new production schema. The baseline example below comes directly from Section 0; an unavailable target-day destination must remain unimplemented and flagged, not silently replaced.

**TODO Varshini:** implement and test `deferred_candidates`, `select_next_deferred`, `select_second_deferred`, `select_target_days_deferred`, `build_destination_schedule` and `apply_destination_rule_to_baseline`. Cover later/eligible candidates, 45/75-session ties (nearer maturity), the second-deferred single-candidate fallback, no-candidate handling and no switches between roll events. Keep the -5-session timing fixed. Open your own issue-linked PR.

In [ ]:
baseline_destination_example = {
    "date": event["date"], "commodity": event["commodity"],
    "outgoing_contract": event["outgoing_contract"],
    "selected_contract": event["baseline_destination"], "fallback_flag": False,
}
pprint(baseline_destination_example)
# TODO: load your tested destination module once it exists in this checkout.
alternative_destination_output = None


### Integration boundary: supply one matched synthetic experiment

**Do not plug the existing fixture cases together directly.** The hand-authored accounting benchmark and the `post_capacity_targets` case are deliberately separate tests. On the first date the accounting benchmark represents $200,000 of prior-settlement notional, whereas the separate post-capacity targets total $1,000,000. Sharing a $1,000,000 return denominator does not make their exposures a matched experiment.

Benchmark and implementation quantities in any P&L/RMSE comparison must be constructed for the **same synthetic experiment, dates, exact-contract universe and valuation data, intended target notionals, and fixed AUM**. This does not require identical realised holdings: destination/timing/capacity may change exposures only as their strand permits, and rounding changes quantities on the supplied contracts. Record how both sides were derived from that one case; identity and timestamp checks alone cannot prove the economic match.

The adapter below is illustrative and stays inactive until all four explicit inputs are supplied. `matched_fixture` supplies the shared calendar, catalogue and observations for the chosen case. `matched_benchmark_rows` and `matched_implementation_rows` supply canonical `Holding` rows for its benchmark and its baseline plus three alternatives. `matched_comparison` is a dictionary containing `dates`, `fixed_aum`, `snapshot_end_exclusive` and `decision_validity` for all portfolios, including the benchmark. No benchmark, dates, AUM or valuation snapshot is silently borrowed from another fixture case.

Each holding's `date` ends `(t-1,t]`; its quantities and inputs must be fixed by the preceding decision cut-off. Supply complete rows, truthful availability flags and explicit zero positions. Never invent decision timestamps from later valuation marks. Call shared accounting once for the full comparison: missing required exact-contract settlements invalidate the common date, without zero-filling or maturity splicing. This is synthetic onboarding only; constructing the historical common benchmark and freezing empirical samples remain Phase B work.

In [ ]:
from src.portfolio.holdings import HoldingsBook
from src.metrics.portfolio_pnl import portfolio_pnl
from src.metrics.tracking_rmse_bps import tracking_rmse_bps

matched_fixture = None  # TODO: shared fixture bundle for ONE chosen synthetic experiment.
matched_benchmark_rows = None  # TODO: its explicitly matched fractional benchmark Holding rows.
matched_implementation_rows = None  # TODO: its baseline + three alternative Holding rows.
matched_comparison = None  # TODO: dates, fixed_aum, snapshot_end_exclusive, decision_validity.
print("Canonical fields:", HOLDINGS_COLUMNS)
if any(value is None for value in (matched_fixture, matched_benchmark_rows,
                                    matched_implementation_rows, matched_comparison)):
    print("TODO: supply a matched synthetic experiment; no accounting comparison is run.")
else:
    benchmark_rows = list(matched_benchmark_rows)
    implementation_rows = list(matched_implementation_rows)
    assert benchmark_rows and implementation_rows, "Supply both sides of the matched experiment."
    assert all(r.strategy == "benchmark" for r in benchmark_rows)
    strategies = ["benchmark"] + sorted({r.strategy for r in implementation_rows})
    assert len(strategies) == len(set(strategies)) == 5, "Provide baseline and three distinct alternatives."
    assert matched_comparison["decision_validity"] is not None, "Supply truthful validity flags."
    book = HoldingsBook(benchmark_rows + implementation_rows,
                        matched_fixture.calendar, matched_fixture.catalogue)
    daily = portfolio_pnl(book, matched_fixture.observations, matched_comparison["dates"],
                          strategies, matched_comparison["fixed_aum"],
                          matched_comparison["snapshot_end_exclusive"],
                          decision_validity=matched_comparison["decision_validity"])
    pprint(tracking_rmse_bps(daily))  # Matched synthetic arithmetic only; no method selection.
